**Group Name:** Course Project Group 25

**Members:** 汪鸿源 72510222

# AIGC Detection - Test Inference

 **Note**: This notebook is used to run inference on the test set and generate prediction results. Best_model.pth files are placed in the checkpoints dir. This notebook is intended solely for inference and integration purposes and does not perform training. The model architecture remains consistent with that in `Train.ipynb` to ensure correctly loaded pre-trained weights.

### 1. 推理环境与模型结构定义

本部分完成：
- 基本依赖导入与设备选择
- 指定测试集与 checkpoint 目录
- 定义与训练阶段完全一致的 `ViTForAIGC` 与 `ResNetForAIGC` 结构（去掉训练相关的组件）
- 定义测试增强与简单 TTA（原图 + 水平翻转）
- 封装 `run_inference` 函数，实现对指定模型的批量推理并输出概率 CSV
- 定义 `ensemble_predictions` 用于对两个模型的预测结果做加权融合

In [1]:
# 推理阶段所需的依赖与基本配置
import os
# 禁止 albumentations 的版本检查，减少无关输出
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
# 使用 hf-mirror.com 作为 HuggingFace 镜像，加速 ViT 权重加载
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import cv2
import glob
import json
from tqdm import tqdm
from transformers import ViTModel, ViTConfig
from torchvision.models import resnet50, ResNet50_Weights
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch.nn.functional as F

# CONFIGURATIONS：设备、测试集与 checkpoint 路径
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TEST_DIR = r'/root/autodl-tmp/5489project/Test_set'           # 无标签测试集目录
CHECKPOINT_DIR = r'/root/autodl-tmp/5489project/checkpoints'  # 训练阶段保存的权重目录

print(f"Using device: {DEVICE}")
print(f"Test directory: {TEST_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

# --- Model Definitions (Must match Train.ipynb) ---
# 与训练 Notebook 中保持完全一致的 ViT-Base 结构，便于加载权重
class ViTForAIGC(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        model_name = "google/vit-base-patch16-224-in21k"
        # pretrained=True 时会从 HuggingFace 下载预训练权重
        self.vit = ViTModel.from_pretrained(model_name) if pretrained else ViTModel(ViTConfig.from_pretrained(model_name))

        if pretrained:
            # 先冻结所有参数
            for param in self.vit.parameters():
                param.requires_grad = False
            # 只解冻 encoder 的后 4 层用于微调
            for i, layer in enumerate(self.vit.encoder.layer):
                if i >= 8:
                    for param in layer.parameters():
                        param.requires_grad = True
        
        # LayerNorm 始终参与训练 / 微调
        for param in self.vit.layernorm.parameters():
            param.requires_grad = True

        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.Linear(self.vit.config.hidden_size, num_classes)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, rgb_images: torch.Tensor):
        # 输入形状 [B, 3, 224, 224]
        outputs = self.vit(pixel_values=rgb_images)
        cls_token_output = outputs.last_hidden_state[:, 0, :]
        cls_token_output = self.dropout(cls_token_output)
        logits = self.classifier(cls_token_output)
        return {"logits": logits}

# 与训练 Notebook 中一致的 ResNet-50 结构
class ResNetForAIGC(nn.Module):
    def __init__(self, num_classes: int = 2, pretrained: bool = True):
        super().__init__()
        # Use standard ResNet-50
        weights = ResNet50_Weights.DEFAULT if pretrained else None
        self.resnet = resnet50(weights=weights)

        # 预训练时冻结大部分层，仅微调最后一个 block（layer4）
        if pretrained:
            for param in self.resnet.parameters():
                param.requires_grad = False
            # Unfreeze the last block (layer4) and fully connected layer
            for param in self.resnet.layer4.parameters():
                param.requires_grad = True
        
        # 替换最终全连接层为 2 类输出
        num_ftrs = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(num_ftrs, num_classes)
        nn.init.xavier_uniform_(self.resnet.fc.weight)
        nn.init.zeros_(self.resnet.fc.bias)

    def forward(self, rgb_images: torch.Tensor):
        logits = self.resnet(rgb_images)
        return {"logits": logits}

# Define transforms：测试阶段图像预处理（无强数据增强）
test_transform_rgb = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# 简单 TTA：原图 + 水平翻转，两次预测取平均
def preprocess_image_tta(image_path: str):
    # 1. 使用 cv2 读取图像
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Failed to load image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # 2. TTA: 原图 + 水平翻转
    images = [image, cv2.flip(image, 1)]
    
    batch_tensors = []
    for img in images:
        augmented = test_transform_rgb(image=img)
        batch_tensors.append(augmented['image'])
    
    # 拼成 batch，形状 [2, 3, 224, 224]
    return torch.stack(batch_tensors).to(DEVICE)

# 针对单个模型进行全量测试集推理并保存概率 CSV
def run_inference(model_name, checkpoint_path, output_csv):
    print(f"\n{'='*20} Processing {model_name} {'='*20}")
    
    if not os.path.exists(checkpoint_path):
        print(f"Checkpoint not found: {checkpoint_path}")
        return False

    # Initialize Model
    print(f"Initializing {model_name}...")
    model = None
    try:
        # 优先使用 pretrained=False，避免重复下载预训练权重
        if model_name == 'ViT-Base':
            model = ViTForAIGC(num_classes=2, pretrained=False)
        elif model_name == 'ResNet50':
            model = ResNetForAIGC(num_classes=2, pretrained=False)
        else:
            raise ValueError(f"Unknown model name: {model_name}")
    except Exception as e:
        # 如果失败则回退为 pretrained=True
        print(f"Warning: Could not initialize with pretrained=False. Trying pretrained=True. Error: {e}")
        if model_name == 'ViT-Base':
            model = ViTForAIGC(num_classes=2, pretrained=True)
        elif model_name == 'ResNet50':
            model = ResNetForAIGC(num_classes=2, pretrained=True)

    if model is None:
        print("Failed to initialize model.")
        return False

    # Load weights：加载训练阶段保存的 state_dict
    print(f"Loading weights from {checkpoint_path}...")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint
        model.load_state_dict(state_dict)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return False

    model = model.to(DEVICE)
    model.eval()
    
    # Retrieve images：收集测试集中所有待预测图片路径
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    test_images = []
    for ext in exts:
        test_images.extend(glob.glob(os.path.join(TEST_DIR, ext)))
    
    if not test_images:
        print(f"No images found in {TEST_DIR}.")
        return False
    else:
        print(f"Found {len(test_images)} images.")
        
    predictions = []  # 存储 fake 概率
    filenames = []    # 存储文件名
    
    with torch.no_grad():
        for img_path in tqdm(test_images, desc=f'Inference {model_name} (TTA)'):
            try:
                # 使用 TTA 预处理
                rgb_batch = preprocess_image_tta(img_path)  # Shape: [2, 3, 224, 224]
                
                outputs = model(rgb_batch)
                logits = outputs['logits']
                probs = F.softmax(logits, dim=1)
                
                # 对 "fake" 类（index 1）的概率在 TTA 维度上取平均
                # probs 形状 [2, 2] -> 取第 1 列 -> [p1, p2] -> mean
                avg_fake_prob = torch.mean(probs[:, 1]).item()
                
                filename = os.path.basename(img_path)
                predictions.append(avg_fake_prob)
                filenames.append(filename)
            except Exception as e:
                # 某些图片处理失败时，用 0.5 作为保守占位概率
                print(f"Error processing {img_path}: {e}")
                filename = os.path.basename(img_path)
                predictions.append(0.5)
                filenames.append(filename)

    # 组装结果并按 ID 排序后写入 CSV
    results_df = pd.DataFrame({
        'ID': filenames,
        'prob': predictions
    })
    results_df['ID'] = results_df['ID'].apply(lambda x: os.path.splitext(x)[0])
    results_df = results_df.sort_values(by='ID').reset_index(drop=True)
    results_df.to_csv(output_csv, index=False)
    print(f"Saved predictions to {output_csv}")
    return True

# 概率级别的加权集成：将 ResNet50 与 ViT-Base 的预测做融合
def ensemble_predictions(resnet_csv, vit_csv, output_csv, w_resnet=0.4, w_vit=0.6):  # Adjusted weighted average
    print(f"\n{'='*20} Running Ensemble {'='*20}")
    if not os.path.exists(resnet_csv) or not os.path.exists(vit_csv):
        print("One or both probability files not found. Skipping ensemble.")
        return

    print(f"Ensembling {resnet_csv} and {vit_csv}...")
    df_resnet = pd.read_csv(resnet_csv)
    df_vit = pd.read_csv(vit_csv)

    # 按 ID 排序并对齐行，确保两文件顺序一致
    df_resnet = df_resnet.sort_values('ID').reset_index(drop=True)
    df_vit = df_vit.sort_values('ID').reset_index(drop=True)

    if not df_resnet['ID'].equals(df_vit['ID']):
        print("Error: IDs do not match between files.")
        return

    df_final = df_resnet.copy()
    # Weighted Average：按给定权重做概率加权
    df_final['prob'] = w_resnet * df_resnet['prob'] + w_vit * df_vit['prob']
    # Convert to label：以 0.5 为阈值将概率转换为标签
    df_final['label'] = (df_final['prob'] > 0.5).astype(int)
    
    submission_df = df_final[['ID', 'label']]
    submission_df.to_csv(output_csv, index=False)
    print(f"Ensemble submission saved to {output_csv}")
    print(f"Prediction distribution: {dict(submission_df['label'].value_counts())}")

Using device: cuda
Test directory: /root/autodl-tmp/5489project/Test_set
Checkpoint directory: /root/autodl-tmp/5489project/checkpoints


### 2. 执行推理与结果集成

本部分顺序执行：
1. 使用 ResNet50 模型对测试集进行推理，生成 `submission_test_ResNet50_prob.csv`
2. 使用 ViT-Base 模型对同一测试集推理，生成 `submission_test_ViT-Base_prob.csv`
3. 对两个模型的概率结果做加权融合，生成最终提交文件 `submission_ensemble.csv`

In [3]:
# 1. Run ResNet50
resnet_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model_ResNet50.pth')
resnet_out = 'submission_test_ResNet50_prob.csv'
run_inference('ResNet50', resnet_ckpt, resnet_out)
    
# 2. Run ViT-Base
vit_ckpt = os.path.join(CHECKPOINT_DIR, 'best_model_ViT-Base.pth')
vit_out = 'submission_test_ViT-Base_prob.csv'
run_inference('ViT-Base', vit_ckpt, vit_out)
    
# 3. Ensemble
ensemble_out = 'submission_ensemble.csv'
ensemble_predictions(resnet_out, vit_out, ensemble_out)


==================== Processing ResNet50 ====================
Initializing ResNet50...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ResNet50.pth...
Weights loaded successfully.
Found 2500 images.
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ResNet50.pth...
Weights loaded successfully.
Found 2500 images.


Inference ResNet50 (TTA): 100%|██████████| 2500/2500 [00:10<00:00, 240.11it/s]


Saved predictions to submission_test_ResNet50_prob.csv

==================== Processing ViT-Base ====================
Initializing ViT-Base...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ViT-Base.pth...
Loading weights from /root/autodl-tmp/5489project/checkpoints/best_model_ViT-Base.pth...
Weights loaded successfully.
Found 2500 images.
Weights loaded successfully.
Found 2500 images.


Inference ViT-Base (TTA): 100%|██████████| 2500/2500 [00:16<00:00, 150.49it/s]

Saved predictions to submission_test_ViT-Base_prob.csv

==================== Running Ensemble ====================
Ensembling submission_test_ResNet50_prob.csv and submission_test_ViT-Base_prob.csv...
Ensemble submission saved to submission_ensemble.csv
Prediction distribution: {1: np.int64(1485), 0: np.int64(1015)}
